# 01 — Basic OpenAI Chat

Goal: Connect to OpenAI API and get the first response.
Learn how messages and responses are structured.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ API key yuklandi:", os.getenv("OPENAI_API_KEY")[:10] + "...")


✅ API key yuklandi: sk-proj-N3...


In [2]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Fransiya Poytaxti haqida  qisqa 2ta gap bilan ma'lumot ber"}
    ]
)

print(response.choices[0].message.content)

Fransiyaning poytaxti Parijs shahri bo‘lib, dunyo madaniyat markazlaridan biridir va Seine daryosining qirg‘oqida joylashgan. Parijs - moda, san'at, qo‘shiqchilik va ma'rifat markazi sifatida mashhur bo‘lib, Eifel minorasi, Luvr muzeyi va Notre-Dam katedrali kabi turistik ajoyib joylari bilan mashhur.


In [3]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Fransiya Poytaxti haqida  qisqa 2ta gap bilan ma'lumot ber"}
    ]
)

print("Javob:", response.choices[0].message.content)
print("\n--- Token sarfi ---")
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

Javob: Fransiyaning poytaxti Parij shahri va bu Fransiya mamlakatining eng katta shahri bo'lib, san'ati, modasi va madaniyati bilan mashhur.

--- Token sarfi ---
Prompt tokens: 32
Completion tokens: 51
Total tokens: 83


In [4]:
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a concise assistant. Answer in only one sentence."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print("Response:", response.choices[0].message.content)
print("Total tokens:", response.usage.total_tokens)

Response: Paris is the capital of France.
Total tokens: 37


In [5]:
#Test 1: Low temperature(deterministic)

response= client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "user", "content": "Give me a creative name for a cofee shop."}

    ]
)
print("Temperature 0:", response.choices[0].message.content)

# Test 2: High temperature (creative)
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=1.5,
    messages=[
        {"role": "user", "content": "Give me a creative name for a coffee shop."}
    ]
)
print("Temperature 1.5:", response.choices[0].message.content)

Temperature 0: Brewed Bliss Café
Temperature 1.5: "Caffeine Haven"


In [14]:
# Test 1: First question
response1 = client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "user", "content": "My name is Abstract."}
    ]
)
print("Response 1:", response1.choices[0].message.content)

# Test 2: Follow-up question
response2 = client.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=0,
    messages=[
        {"role": "user", "content": "What is my name?"}
    ]
)
print("Response 2:", response2.choices[0].message.content)

Response 1: Hello Abstract, nice to meet you! How can I assist you today?
Response 2: I'm sorry, I do not have access to personal information such as your name.


In [13]:
class ChatBot:
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
    
    def chat(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0.7,
            messages=self.messages
        )
        
        assistant_reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_reply})
        
        return assistant_reply

In [15]:
# Create a new bot
bot = ChatBot(system_prompt="You are a friendly assistant. Keep answers short.")

# First message
print("User: My name is Abstract.")
print("Bot:", bot.chat("My name is Abstract."))
print()

# Second message — bot should remember
print("User: What is my name?")
print("Bot:", bot.chat("What is my name?"))
print()

# Third message
print("User: What did I just ask you?")
print("Bot:", bot.chat("What did I just ask you?"))

User: My name is Abstract.
Bot: Nice to meet you, Abstract! How can I assist you today?

User: What is my name?
Bot: Your name is Abstract.

User: What did I just ask you?
Bot: You asked me what your name is.


In [16]:
# Look inside the bot's memory
print(f"Total messages in memory: {len(bot.messages)}\n")

for i, msg in enumerate(bot.messages):
    print(f"{i}. [{msg['role']}]: {msg['content']}")

Total messages in memory: 7

0. [system]: You are a friendly assistant. Keep answers short.
1. [user]: My name is Abstract.
2. [assistant]: Nice to meet you, Abstract! How can I assist you today?
3. [user]: What is my name?
4. [assistant]: Your name is Abstract.
5. [user]: What did I just ask you?
6. [assistant]: You asked me what your name is.


In [17]:
class ChatBotWithTracking:
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.messages = [
            {"role": "system", "content": system_prompt}
        ]
        self.total_tokens = 0  # NEW: track tokens
    
    def chat(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0.7,
            messages=self.messages
        )
        
        # Track tokens
        tokens_this_call = response.usage.total_tokens
        self.total_tokens += tokens_this_call
        
        assistant_reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_reply})
        
        # Print stats
        print(f"  [Tokens this call: {tokens_this_call} | Total so far: {self.total_tokens}]")
        
        return assistant_reply

In [18]:
bot = ChatBotWithTracking(system_prompt="You are a helpful assistant. Keep answers short.")

print("Q1:", bot.chat("My name is Abstract."))
print("Q2:", bot.chat("What is my name?"))
print("Q3:", bot.chat("What did I just ask you?"))
print("Q4:", bot.chat("And before that?"))
print("Q5:", bot.chat("Tell me a fun fact about Paris."))

print(f"\n=== TOTAL TOKENS USED: {bot.total_tokens} ===")

  [Tokens this call: 40 | Total so far: 40]
Q1: Nice to meet you, Abstract. How can I assist you today?
  [Tokens this call: 58 | Total so far: 98]
Q2: Your name is Abstract.
  [Tokens this call: 79 | Total so far: 177]
Q3: You asked about your name.
  [Tokens this call: 97 | Total so far: 274]
Q4: You introduced yourself as Abstract.
  [Tokens this call: 136 | Total so far: 410]
Q5: Paris is often called the "City of Light" because it was one of the first cities to have street lighting.

=== TOTAL TOKENS USED: 410 ===


In [19]:
class ChatBotSliding:
    def __init__(self, system_prompt="You are a helpful assistant.", max_history=4):
        self.system_message = {"role": "system", "content": system_prompt}
        self.messages = []  # only user/assistant, no system
        self.max_history = max_history  # how many messages to keep
        self.total_tokens = 0
    
    def chat(self, user_message):
        self.messages.append({"role": "user", "content": user_message})
        
        # Build full list: system + last N messages
        messages_to_send = [self.system_message] + self.messages[-self.max_history:]
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=0.7,
            messages=messages_to_send
        )
        
        tokens_this_call = response.usage.total_tokens
        self.total_tokens += tokens_this_call
        
        assistant_reply = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_reply})
        
        print(f"  [Sent {len(messages_to_send)} msgs | Tokens: {tokens_this_call} | Total: {self.total_tokens}]")
        
        return assistant_reply

In [20]:
bot = ChatBotSliding(system_prompt="You are a helpful assistant. Keep answers short.", max_history=4)

print("Q1:", bot.chat("My name is Abstract."))
print("Q2:", bot.chat("I live in Fergana."))
print("Q3:", bot.chat("I am learning AI."))
print("Q4:", bot.chat("What is my name?"))      # ← bot esladingmi?
print("Q5:", bot.chat("Where do I live?"))      # ← bot esladingmi?

print(f"\n=== TOTAL TOKENS: {bot.total_tokens} ===")

  [Sent 2 msgs | Tokens: 40 | Total: 40]
Q1: Nice to meet you, Abstract. How can I assist you today?
  [Sent 4 msgs | Tokens: 75 | Total: 115]
Q2: That's great. Is there anything specific you would like to know or do in Fergana?
  [Sent 5 msgs | Tokens: 110 | Total: 225]
Q3: That's wonderful! AI is a fascinating field. If you have any questions or need resources to help you learn more about AI, feel free to ask.
  [Sent 5 msgs | Tokens: 117 | Total: 342]
Q4: I'm sorry, I don't have access to personal information like your name. How can I assist you today?
  [Sent 5 msgs | Tokens: 120 | Total: 462]
Q5: I'm sorry, I don't have access to personal information like your location. How can I assist you today?

=== TOTAL TOKENS: 462 ===
